<br>

# Notebook II — Crop Simulation
<hr>

## Overview
This notebook runs **Module II** — the most compute-intensive step in the PyAEZ pipeline. For every pixel inside the Pakistan admin mask it simulates **365 possible crop-cycle starting dates** and selects the one that produces the highest yield.

### Prerequisites
NB1 must have been run first. This notebook reads:
- Climate `.npy` arrays from `data_input/climate/`
- Spatial rasters from `data_input/`
- NB1 outputs: `data_output/NB1/PK_LGP.tif`, `PK_LGPt5.tif`, `PK_LGPt10.tif`

### What this notebook produces (saved to `data_output/NB2/`)
| Output file | Description |
|---|---|
| `maiz_yield_rain.tif` | Maximum attainable rainfed yield (kg/ha) |
| `maiz_yield_irr.tif` | Maximum attainable irrigated yield (kg/ha) |
| `maiz_starting_date_rain.tif` | Optimum sowing date — rainfed (day of year 1–365) |
| `maiz_starting_date_irr.tif` | Optimum sowing date — irrigated (day of year 1–365) |
| `fc1_maiz_rain/irr.tif` | Thermal screening reduction factor (0–1) |
| `fc2_maiz_rain.tif` | Moisture reduction factor for rainfed (0–1) |

### ⚠ Runtime warning
`simulateCropCycle` runs pixel-by-pixel and can take **1–3 hours** on the full Pakistan grid (87×108 pixels × 365 start dates). Numba JIT compilation adds ~1 min on first run.

Prepared by Geoinformatics Center, AIT — Pakistan adaptation by Saqib Ali
<hr>

### Step 1 — Connect to Google Drive (Colab only)

Mount Google Drive so the notebook can read climate data and NB1 outputs from `/content/drive/MyDrive/PK-PyAEZ/`.

> **Skip** if running locally.

In [1]:


# Step 2 (optional): Mount Google Drive to persist outputs across sessions
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys, os
repo = '/content/drive/MyDrive/PK-PyAEZ'
if repo not in sys.path:
    sys.path.insert(0, repo)
os.chdir(repo)

### Step 2 — Install dependencies (Colab only)

Installs GDAL, Numba, and all required Python packages, then installs `pyaez` in editable mode from the repo so any changes to `pyaez/*.py` take immediate effect.

> **Skip** if running locally with conda environment already configured.

In [ ]:
import sys

if 'google.colab' in sys.modules:
    import subprocess, os as _os
    subprocess.run(['apt-get', 'install', '-y', '-q', 'gdal-bin', 'libgdal-dev', 'python3-gdal'],
                   capture_output=True)
    _ver = subprocess.check_output(['gdal-config', '--version']).decode().strip()
    subprocess.run(['pip', 'install', '-q',
                    f'GDAL=={_ver}', 'numba', 'openpyxl', 'rasterio', 'requests', 'scipy', 'pandas'],
                   capture_output=True)
    subprocess.run(['pip', 'install', '-q', '-e', '/content/drive/MyDrive/PK-PyAEZ'], capture_output=True)
    print(f"Colab setup complete — GDAL {_ver}")

### Step 3 — Import Python libraries

| Library | Purpose |
|---|---|
| `numpy` | Loads `.npy` climate arrays; array math |
| `matplotlib` | Plots yield maps and crop calendars |
| `gdal` (osgeo) | Opens input rasters; saves outputs as GeoTIFF |
| `os`, `sys` | File path and module path management |

`gdal.UseExceptions()` silences the GDAL 4.0 FutureWarning.

In [ ]:
'''import supporting libraries'''
import numpy as np
import matplotlib.pyplot as plt
import os
try:
    from osgeo import gdal
except:
    import gdal
gdal.UseExceptions()
import sys

### Step 4 — Set working directory

Sets the current working directory to the **repo root** so all `./data_input/` and `./data_output/` paths resolve correctly.

- **Colab**: `/content/drive/MyDrive/PK-PyAEZ`
- **Local**: `..` (one level up from `tutorials/`)

In [ ]:
'Set the working directory'
import sys, os

if 'google.colab' in sys.modules:
    work_dir = '/content/drive/MyDrive/PK-PyAEZ'
else:
    work_dir = '..'

os.chdir(work_dir)
sys.path.insert(0, os.getcwd())
os.getcwd()

### Step 5 — Create output folder

Creates `data_output/NB2/` if it does not already exist. All yield maps, crop calendars, and reduction factors from this notebook are saved there.

In [6]:
import os
folder_path = './data_output/NB2/'
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print("Folder created successfully.")
else:
    print("Folder already exists.")


Folder already exists.


<hr>

## MODULE II — Crop Simulation

### Step 6 — Initialise the CropSimulation object

Creates the `aez` (CropSimulation) and `obj_util` (UtilitiesCalc) instances.

`CropSimulation` is the core engine. After data is loaded into it, it:
1. Runs the **De Wit (1965) biomass model** for each pixel and each of 365 starting dates
2. Applies **Penman-Monteith ETo** and **FAO CropWat** water balance
3. Returns the starting date that maximises yield at each pixel

In [7]:
# Import Module 2 and initate Class intance

from pyaez import CropSimulation, UtilitiesCalc
aez = CropSimulation.CropSimulation()
obj_util = UtilitiesCalc.UtilitiesCalc()

### Step 7 — Load climate and spatial data

Same six climate arrays as NB1, plus the admin mask and elevation raster.

| Variable | File | Unit |
|---|---|---|
| `max_temp` | `climate/max_temp.npy` | °C |
| `min_temp` | `climate/min_temp.npy` | °C |
| `precipitation` | `climate/precipitation.npy` | mm/month |
| `short_rad` | `climate/short_rad.npy` | W/m² |
| `wind_speed` | `climate/wind_speed.npy` | m/s |
| `rel_humidity` | `climate/relative_humidity.npy` | fraction 0–1 |
| `mask` | `PK_Admin.tif` | binary (1 = Pakistan) |
| `elevation` | `PK_Elevation.tif` | metres |

All arrays must have the same spatial shape **(87 rows × 108 cols)**.

In [8]:
'''reading climate data'''
# Importing the climate data
max_temp = np.load('./data_input/climate/max_temp.npy')# maximum temperature
min_temp = np.load('./data_input/climate/min_temp.npy')  # minimum temperature
precipitation = np.load('./data_input/climate/precipitation.npy')  # precipitation
rel_humidity = np.load('./data_input/climate/relative_humidity.npy') # relative humidity
wind_speed = np.load('./data_input/climate/wind_speed.npy')# wind speed measured at two meters
short_rad = np.load('./data_input/climate/short_rad.npy') # shortwave radiation

# Load the geographical data/rasters
mask_path = './data_input/PK_Admin.tif'
mask = gdal.Open(mask_path).ReadAsArray()
elevation = gdal.Open('./data_input/PK_Elevation.tif').ReadAsArray()


/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


### Step 8 — Set analysis parameters

| Parameter | Value | Description |
|---|---|---|
| `lat_min` | `23.0` | Southern boundary of Pakistan (°N) |
| `lat_max` | `37.5` | Northern boundary of Pakistan (°N) |
| `mask_value` | `0` | Pixel value in admin mask to exclude from simulation |
| `daily` | `False` | `False` = monthly WorldClim inputs (interpolated to daily internally) |

In [9]:
# Define the Area-Of-Interest's geographical extents
lat_min = 23.0   # Pakistan southern boundary
lat_max = 37.5   # Pakistan northern boundary
mask_value = 0  # pixel value in admin_mask to exclude from the analysis
daily = False #Type of climate data = True: daily, False: monthly

### Step 9 — Load data into CropSimulation

Passes climate arrays, admin mask, and elevation into the `aez` object. Internally converts monthly arrays to daily resolution and sets up the Penman-Monteith ETo calculation — same as NB1.

In [10]:
aez.setStudyAreaMask(mask, mask_value)
aez.setLocationTerrainData(lat_min, lat_max, elevation)
if daily:
    aez.setDailyClimateData(
        min_temp, max_temp, precipitation, short_rad, wind_speed, rel_humidity)
else:
    aez.setMonthlyClimateData(
        min_temp, max_temp, precipitation, short_rad, wind_speed, rel_humidity)


### Step 10 — Set crop parameters (mandatory)

Reads crop-specific parameters from an Excel file:

| Parameter set | File | Description |
|---|---|---|
| Crop cycle & TSUM | `input_crop_TSUM_parameters_maiz_sugar.xlsx` | Base temperature, TSUM1, TSUM2, cycle length, biomass-yield ratio |
| Soil water | `Sa`, `pc` | Available soil water (mm); soil water depletion fraction |

**`crop_name`** selects the sheet within the Excel file (`'maize'` or `'sugarcane'`).

**`setSoilWaterParameters(Sa, pc)`** — `Sa=100 mm` is a typical value for medium-texture soils. Adjust for your soil data if available.

In [11]:
# setting up the crop parameters, crop cycle and soil water parameters ***mandatory step
# New function, reading crop-specific biomass/yield loss/TSUM screening factors from excel sheet, xlsx file.
aez.readCropandCropCycleParameters(file_path = './data_input/input_crop_TSUM_parameters_maiz_sugar.xlsx',
                                   crop_name = 'maize')


aez.setSoilWaterParameters(Sa= 100*np.ones((mask.shape)), pc=0.5)

### Step 11 — Thermal screening (optional, but recommended for perennials)

Applies two additional temperature-based filters before simulation:

#### Type A — Thermal Climate Screening
Excludes pixels in thermal climate classes where the crop cannot survive (e.g. classes 2, 6–12 for maize). Set `no_t_climate` to a list of classes to mask out.

#### Type B — Crop-Specific Rule Screening
Reads temperature profile rules from `crop-specific_rule_maiz_sugar.xlsx`. For perennials like sugarcane this checks multi-year temperature patterns.

#### Permafrost Screening
Excludes pixels with continuous or discontinuous permafrost (classes 1–2) where frost damage would prevent cropping.

> **`is_perennial = True`** — set to `False` for annual crops like maize to skip the perennial-specific checks.

In [12]:
# If you're simulating perennial crops, this thermal screening is mandatory
# Compute Thermal Climate
tclimate = aez.getThermalClimate()

# Compute permafrost
permafrost_eval = aez.AirFrostIndexandPermafrostEvaluation()
frost_index = permafrost_eval[0]
permafrost_class = permafrost_eval[1]
# tclimate = gdal.Open("./data_output/NB1/PK_ThermalClimate.tif").ReadAsArray()# User to change this TClimate file
# permafrost_class = gdal.Open("./data_output/NB1/PK_permafrost.tif").ReadAsArray()# User to change this permafrost file

# Thermal Climate screening
aez.setThermalClimateScreening(tclimate, no_t_climate=[2,6,7,8,9,10,11,12])

# New Thermal Screening: Permafrost Screening
aez.setPermafrostScreening(permafrost_class= permafrost_class)

# Updated Temperature Profile screenign routine
aez.setCropSpecificRule(file_path = './data_input/crop-specific_rule_maiz_sugar.xlsx',
                         crop_name = 'maize')

# Is the crop perennial? (True/False)
is_perennial = True

/content/drive/MyDrive/PK-PyAEZ/PK-PyAEZ/pyaez/CropSimulation.py:1223: RuntimeWarning: invalid value encountered in divide
  fi = np.sqrt(ddf)/(np.sqrt(ddf) + np.sqrt(ddt))


### Step 12 — Import LGP layers from NB1

Loads the moisture-based LGP and thermal LGP outputs computed in NB1. These are required by the crop simulation to determine:
- Which pixels have a long enough growing season for the crop
- Whether moisture conditions support rainfed production

| Variable | Source file | Used for |
|---|---|---|
| `lgp` | `PK_LGP.tif` | Moisture-based growing period (days) |
| `lgp5` | `PK_LGPt5.tif` | Thermal LGP > 5 °C (days) |
| `lgp10` | `PK_LGPt10.tif` | Thermal LGP > 10 °C (days) |

> **NB1 must be run first.** The cell above verifies these files exist before loading.

In [ ]:
import os, sys
repo = '/content/drive/MyDrive/PK-PyAEZ/'
os.chdir(repo)

# Verify NB1 outputs exist
for f in ['data_output/NB1/PK_LGP.tif', 'data_output/NB1/PK_LGPt5.tif', 'data_output/NB1/PK_LGPt10.tif']:
    print(f, "✓" if os.path.exists(f) else "MISSING")

data_output/NB1/PK_LGP.tif ✓
data_output/NB1/PK_LGPt5.tif ✓
data_output/NB1/PK_LGPt10.tif ✓


In [14]:
# Create climatic indicators independently in this notebook
# lgpt5 = aez.getThermalLGP5()
# lgpt10 = aez.getThermalLGP10()
# lgp = aez.getLGP(Sa=100, D=1) #has to be after LGPt are computed

# Or if you have the agro-climatic indicators calculated in Module I, you can use them.
lgp = gdal.Open('./data_output/NB1/PK_LGP.tif').ReadAsArray()
lgp5 = gdal.Open('./data_output/NB1/PK_LGPt5.tif').ReadAsArray()
lgp10 = gdal.Open('./data_output/NB1/PK_LGPt10.tif').ReadAsArray()

aez.ImportLGPandLGPT(lgp = lgp, lgpt5 = lgp5, lgpt10= lgp10)

### Step 13 — Run crop cycle simulation ⚠ Long-running

This is the main computation. For every pixel in the mask, `simulateCropCycle` loops through all `start_doy` values and simulates the full crop growth cycle using:
1. **BioMassCalc** — De Wit (1965) gross biomass production from radiation
2. **CropWatCalc** — FAO water balance; applies moisture stress for rainfed
3. **ThermalScreening** — applies fc1 (thermal) and optional Type B rules

| Parameter | Value | Description |
|---|---|---|
| `start_doy` | `1` | First day-of-year to simulate |
| `end_doy` | `365` | Last day-of-year to simulate |
| `step_doy` | `1` | Step between starting dates (increase to speed up; reduces resolution) |
| `leap_year` | `False` | Set `True` if your data represents a leap year |

> **Runtime:** 1–3 hours on the full Pakistan grid. To test quickly, set `step_doy=10` (reduces to ~36 starting dates per pixel).

In [15]:
'''run simulations'''
aez.simulateCropCycle( start_doy=1, end_doy=365, step_doy=1, leap_year=False) # results are in kg / hectare

KeyboardInterrupt: 

### Step 14 — Extract results

Extracts the maximum yield and optimum sowing date from the simulation:

| Variable | Method | Description |
|---|---|---|
| `yield_map_rain` | `getEstimatedYieldRainfed()` | Max rainfed yield (kg/ha) across all start dates |
| `yield_map_irr` | `getEstimatedYieldIrrigated()` | Max irrigated yield (kg/ha) |
| `starting_date_rain` | `getOptimumCycleStartDateRainfed()` | Day-of-year that gave max rainfed yield |
| `starting_date_irr` | `getOptimumCycleStartDateIrrigated()` | Day-of-year that gave max irrigated yield |

**Yield suitability classes** (optional, via `classifyFinalYield`):

| Class | Yield relative to maximum |
|---|---|
| 1 — Not suitable | 0–20% |
| 2 — Marginally suitable | 20–40% |
| 3 — Moderately suitable | 40–60% |
| 4 — Suitable | 60–80% |
| 5 — Very suitable | > 80% |

In [ ]:
# Now, showing the estimated and highly obtainable yield of a particular crop, results are in kg / hectare
yield_map_rain = aez.getEstimatedYieldRainfed()  # for rainfed
yield_map_irr = aez.getEstimatedYieldIrrigated()  # for irrigated

# Optimum cycle start date, the date when the highest yield are produced referenced from the start of crop cycle
starting_date_rain = aez.getOptimumCycleStartDateRainfed()
starting_date_irr = aez.getOptimumCycleStartDateIrrigated()

## get classified output of yield
# yield_map_rain_class = obj_util.classifyFinalYield(yield_map_rain)
# yield_map_irr_class = obj_util.classifyFinalYield(yield_map_irr)


In [ ]:
'''visualize result'''

"""Yield Maps"""
plt.imshow(yield_map_rain, vmax = np.max([yield_map_irr, yield_map_rain]), vmin = 0)
plt.colorbar()
plt.title('Rainfed Yield')
plt.show()

plt.imshow(yield_map_irr, vmax = np.max([yield_map_irr, yield_map_rain]), vmin = 0)
plt.colorbar()
plt.title('Irrigated Yield')
plt.show()

"""Starting Date (Crop Calendar)"""
plt.imshow(starting_date_rain, vmin= 0, vmax = 366)
plt.colorbar()
plt.title('Starting Date Rainfed')
plt.show()

plt.imshow(starting_date_irr, vmin= 0, vmax = 366)
plt.colorbar()
plt.title('Starting Date Irrigated')
plt.show()

"""Classified Yield"""
# plt.imshow(yield_map_rain_class)
# plt.colorbar()
# plt.title('Classified Rainfed Yield')
# plt.show()
# plt.imshow(yield_map_irr_class)
# plt.colorbar()
# plt.show('Classified Irrigated Yield')
# plt.show()

In [ ]:
# '''save result'''

obj_util.saveRaster(mask_path, './data_output/NB2/maiz_yield_rain.tif', yield_map_rain)
obj_util.saveRaster(mask_path, './data_output/NB2/maiz_yield_irr.tif', yield_map_irr)

obj_util.saveRaster(mask_path, './data_output/NB2/maiz_starting_date_rain.tif', starting_date_rain)
obj_util.saveRaster(mask_path, './data_output/NB2/maiz_starting_date_irr.tif', starting_date_irr)

# obj_utilities.saveRaster(mask_path, r'.\data_output\NB2\maiz_yld_rain_class.tif',yield_map_rain_class)
# obj_utilities.saveRaster(mask_path, r'.\data_output\NB2\maiz_yld_irr_class.tif',yield_map_irr_class)

---
### Thermal Screening Factor (fc1)

`fc1` is the **thermal reduction factor** — a value between 0 and 1 that indicates what fraction of the potential yield can be achieved given the temperature constraints at each pixel.

- `fc1 = 1.0` → no thermal limitation
- `fc1 = 0.0` → pixel is thermally unsuitable (excluded by screening)

Computed separately for rainfed and irrigated (same thermal constraint, different moisture).

- **Output:** `fc1_rain`, `fc1_irr`
- **Saved to:** `data_output/NB2/fc1_maiz_rain.tif`, `fc1_maiz_irr.tif`
- **Used by:** NB3 climatic constraints (applied on top of fc1)

In [ ]:
screen_factor = aez.getThermalReductionFactor()
fc1_rain = screen_factor[0]
fc1_irr = screen_factor[1]

In [ ]:
'''visualize additional outputs (Fc1)'''

plt.imshow(fc1_rain, vmax = 1, vmin = 0)
plt.colorbar()
plt.title('Fc1 Rainfed')
plt.show()

plt.imshow(fc1_irr, vmax = 1, vmin = 0)
plt.colorbar()
plt.title('Fc1 Irrigated')
plt.show()

In [ ]:
# '''save fc1 result'''
obj_util.saveRaster(mask_path, './data_output/NB2/fc1_maiz_rain.tif', fc1_rain)
obj_util.saveRaster(mask_path, './data_output/NB2/fc1_maiz_irr.tif', fc1_irr)

---
### Moisture Reduction Factor (fc2) — Rainfed only

`fc2` captures the **moisture stress** experienced by the crop under rainfed conditions. It compares actual water supply (precipitation) to crop water demand (ETo × crop coefficient).

- `fc2 = 1.0` → no moisture limitation
- `fc2 < 1.0` → yield reduced due to water deficit

> fc2 is **not applied for irrigated** conditions (assumed full water supply).

- **Output:** `fc2`
- **Saved to:** `data_output/NB2/fc2_maiz_rain.tif`
- **Used by:** NB3 climatic constraints

In [ ]:
fc2 = aez.getMoistureReductionFactor()

In [ ]:
""" visualize additional outputs (Fc2) """

plt.imshow(fc2, vmin= 0, vmax = 1)
plt.colorbar()
plt.title('fc2 (Moisture limited reduction factor)')
plt.show()

In [ ]:
# saving fc2 result
obj_util.saveRaster(mask_path, './data_output/NB2/fc2_maiz_rain.tif', fc2)

<hr>

### END OF MODULE II — Crop Simulation

**Next step → NB3:** Apply climatic constraints (fc3) to the yield maps produced here.

Inputs passed forward to NB3:
- `data_output/NB2/maiz_yield_rain.tif`
- `data_output/NB2/maiz_yield_irr.tif`

<hr>